# Piecewise Conversion (Part-Load Curves)

Most real converters do not have a constant efficiency. A condensing boiler
loses efficiency at low load (cycling losses) and at high load (higher flue
temperatures); a heat pump's COP varies with ambient temperature; a CHP
delivers a non-constant heat-to-power ratio across its load range.

`ConversionCurve` ties two or more flows together along a piecewise-linear
curve. The solver picks an operating point on the curve at every timestep.

**New concepts**: ConversionCurve (piecewise-linear input/output coupling · multi-flow joint curves · method auto-dispatch · status-gated curves)

In [ ]:
from datetime import datetime, timedelta

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from fluxopt import Carrier, ConversionCurve, Converter, Effect, Flow, Port, Status, optimize

pio.renderers.default = 'notebook_connected'

## The Boiler Curve

A 100 kW condensing boiler with three efficiency segments:

| Load range | Slope (η) | Why |
|---|---|---|
| 0–30 kW gas | 0.75 | Cycling / standby losses dominate |
| 30–70 kW gas | 0.90 | Sweet spot — condensing returns are flue-cool |
| 70–100 kW gas | 0.67 | Flue gases run hotter, less condensation |

Breakpoints (`gas`, `heat`) = `[0, 30, 70, 100]`, `[0, 22.5, 58.5, 78.5]`.

In [ ]:
fuel_bp = [0, 30, 70, 100]
heat_bp = [0, 22.5, 58.5, 78.5]

fig = go.Figure()
fig.add_trace(go.Scatter(x=fuel_bp, y=heat_bp, mode='lines+markers', name='curve', line_color='#636efa'))
fig.update_layout(
    height=300,
    margin={'l': 50, 'r': 20, 't': 10, 'b': 40},
    template='plotly_white',
    xaxis_title='gas input [kW]',
    yaxis_title='heat output [kW]',
)
fig

## Input Data — 24 h with mixed-load demand

Demand spans the whole load range so the solver visits every segment of the curve.

In [ ]:
n = 24
timesteps = [datetime(2024, 1, 15) + timedelta(hours=h) for h in range(n)]

demand = np.array(
    [
        10,
        10,
        15,
        20,
        25,
        35,  # night: low load → cheap segment then sweet spot
        50,
        65,
        70,
        60,
        50,
        45,  # morning: middle of curve
        40,
        35,
        30,
        25,
        20,
        30,  # midday: lower load
        55,
        70,
        78,
        75,
        60,
        35,  # evening peak: hits the inefficient high-load segment
    ]
)

fig = go.Figure(go.Bar(x=timesteps, y=demand, marker_color='#ef553b'))
fig.update_layout(
    height=250,
    margin={'l': 50, 'r': 20, 't': 10, 'b': 20},
    template='plotly_white',
    yaxis_title='heat demand [kW]',
)
fig

## System

```
Gas Grid ──► [gas] ──► Boiler (piecewise) ──► [heat] ──► Building
 0.05 €/kWh         3-segment curve
```

The `ConversionCurve` is passed as `conversion=` on the `Converter`. The
dict form `{flow_short_id: breakpoints}` ties every listed flow to the
same set of interpolation weights with equality bounds. With only two
flows, an inequality bound (`'>='`) on one of them would unlock the LP
fast path — useful for convex/concave fuel curves where the solver only
needs an outer bound.

The `method='auto'` (default) lets linopy pick the formulation:
`lp` for matching-curvature inequality curves, `incremental` for
monotonic breakpoints, otherwise `sos2`.

In [ ]:
result = optimize(
    timesteps=timesteps,
    carriers=[Carrier('gas'), Carrier('heat')],
    effects=[Effect('cost', unit='EUR')],
    objective_effects='cost',
    ports=[
        Port('gas_grid', imports=[Flow('gas', size=500, effects_per_flow_hour={'cost': 0.05})]),
        Port(
            'building',
            exports=[Flow('heat', size=float(demand.max()), fixed_relative_profile=demand / demand.max())],
        ),
    ],
    converters=[
        Converter(
            'boiler',
            inputs=[Flow('gas', short_id='fuel')],
            outputs=[Flow('heat', size=100)],
            conversion=ConversionCurve({'fuel': fuel_bp, 'heat': heat_bp}),
        ),
    ],
)

print(f'Total cost: {result.objective:.2f} EUR')

## Operating Points on the Curve

Each timestep places the boiler at one point on the (gas, heat) curve.
The dots show *where* the solver landed; their density along each
segment tells you which efficiency regime the boiler ran in.

In [ ]:
gas_rate = result.flow_rate('boiler(fuel)').values
heat_rate = result.flow_rate('boiler(heat)').values

fig = go.Figure()
fig.add_trace(go.Scatter(x=fuel_bp, y=heat_bp, mode='lines+markers', name='curve', line_color='#636efa'))
fig.add_trace(
    go.Scatter(
        x=gas_rate,
        y=heat_rate,
        mode='markers',
        name='operating points',
        marker={'symbol': 'x', 'size': 9, 'color': '#ef553b'},
    )
)
fig.update_layout(
    height=350,
    margin={'l': 50, 'r': 20, 't': 10, 'b': 40},
    template='plotly_white',
    xaxis_title='gas input [kW]',
    yaxis_title='heat output [kW]',
)
fig

In [ ]:
times = result.flow_rates.coords['time'].values

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=('Heat & Gas Flows', 'Instantaneous Efficiency'),
)
fig.add_trace(
    go.Scatter(x=times, y=heat_rate, name='heat out', line_shape='hv', line_color='#636efa'),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=times, y=gas_rate, name='gas in', line_shape='hv', line_color='#ef553b'),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=times, y=demand, name='demand', line_shape='hv', line={'color': 'black', 'dash': 'dot'}),
    row=1,
    col=1,
)

eta = np.where(gas_rate > 1e-6, heat_rate / np.where(gas_rate > 1e-6, gas_rate, 1), np.nan)
fig.add_trace(
    go.Scatter(x=times, y=eta, name='η = heat/gas', line_shape='hv', line_color='#00cc96'),
    row=2,
    col=1,
)
fig.update_layout(
    height=500,
    margin={'l': 50, 'r': 20, 't': 30, 'b': 20},
    template='plotly_white',
)
fig.update_yaxes(title_text='kW', row=1, col=1)
fig.update_yaxes(title_text='efficiency', row=2, col=1, range=[0, 1])
fig

## Adding On/Off Behaviour

Pass a `Status` to `ConversionCurve` to gate the entire curve with one
binary. When `on=0`, every curve flow is pinned to its first breakpoint
(typically zero), so the converter is fully off — no gas in, no heat out.
Without status, `add_piecewise_formulation` only enforces that the
operating point lies on the curve; (0, 0) being one of the breakpoints
is usually enough — but startup costs, minimum uptime, and similar
constraints all need an explicit binary.

In [ ]:
result_status = optimize(
    timesteps=timesteps,
    carriers=[Carrier('gas'), Carrier('heat')],
    effects=[Effect('cost', unit='EUR')],
    objective_effects='cost',
    ports=[
        Port('gas_grid', imports=[Flow('gas', size=500, effects_per_flow_hour={'cost': 0.05})]),
        Port(
            'backup',
            imports=[Flow('heat', size=200, effects_per_flow_hour={'cost': 0.20})],
        ),
        Port(
            'building',
            exports=[Flow('heat', size=float(demand.max()), fixed_relative_profile=demand / demand.max())],
        ),
    ],
    converters=[
        Converter(
            'boiler',
            inputs=[Flow('gas', short_id='fuel')],
            outputs=[Flow('heat', size=100)],
            conversion=ConversionCurve(
                {'fuel': fuel_bp, 'heat': heat_bp},
                status=Status(
                    min_uptime=3,
                    effects_per_startup={'cost': 20},
                    effects_per_running_hour={'cost': 2},
                ),
            ),
        ),
    ],
)

print(f'Total cost (with status): {result_status.objective:.2f} EUR')

In [ ]:
on = result_status.solution['component--on'].sel(component='boiler').values
heat_main = result_status.flow_rate('boiler(heat)').values
heat_backup = result_status.flow_rate('backup(heat)').values

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=('Heat Supply', 'Boiler On/Off'),
)
fig.add_trace(go.Scatter(x=times, y=heat_main, name='boiler', line_shape='hv', line_color='#636efa'), row=1, col=1)
fig.add_trace(go.Scatter(x=times, y=heat_backup, name='backup', line_shape='hv', line_color='#ef553b'), row=1, col=1)
fig.add_trace(
    go.Scatter(x=times, y=demand, name='demand', line_shape='hv', line={'color': 'black', 'dash': 'dot'}),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=times, y=on, name='on', line_shape='hv', fill='tozeroy', line_color='#636efa'),
    row=2,
    col=1,
)
fig.update_layout(height=450, margin={'l': 50, 'r': 20, 't': 30, 'b': 20}, template='plotly_white')
fig.update_yaxes(title_text='kW', row=1, col=1)
fig.update_yaxes(title_text='on/off', row=2, col=1, range=[-0.1, 1.2])
fig

## Insights

- **Operating points cluster on the sweet spot**: when demand is between
  ~22 and ~58 kW, the solver places the boiler on the middle (η=0.90)
  segment. The cheap segment is wherever the solver can land for free.

- **High evening peak forces the inefficient segment**: at the evening
  peak the demand exceeds 58 kW, so the operating point slides into the
  third segment (η=0.67). The instantaneous efficiency plot drops there.

- **One curve, three flows is just as easy**: pass three (or more)
  breakpoint vectors of the same length and they are all tied to the
  same interpolation weights — useful for CHP plants where the heat-to-
  power ratio varies with load.

- **Status gating is per-curve**: with `ConversionCurve(..., status=Status(...))`,
  the binary applies to the entire converter — you cannot also put a
  `Status` on individual flows of a piecewise converter (use one or the
  other).

- **Method auto-dispatch**: the default `method='auto'` lets `linopy`
  pick the cheapest formulation. For our 3-segment equality curve it
  uses `incremental` (one binary per piece). A 2-flow curve with one
  inequality flow and matching curvature would skip binaries entirely
  via `lp`.